# 19 · Prompting that works
Level L2 · Part 3 · core · ~30 min · builds on 07

## Goal
Rewrite Scout's vague note-summary prompt using clear instructions, a role, XML tags, three examples and
data-first ordering, then measure both prompts on the same 5 notes with a checker that counts how many
replies Scout can actually use.

## The idea
A model only knows what is in the request. "Summarise this note" leaves it to guess the length, the
format and who the summary is for, so each reply guesses differently. A good prompt removes the guessing:
it says exactly what to produce and why, gives the model a **role**, wraps each part in **XML tags** so
instructions and data cannot blur, shows **3 to 5 examples** of a correct answer, and puts the data
first and the question last. None of this is magic. You test it the way you test code: run it on
several inputs and count passes.

Analogy: briefing a new colleague. "Write something about this" gets you anything. A job title, a
template, three finished samples and the rule "one line each" get you what you need.

## Picture
```mermaid
flowchart TB
    subgraph Weak["weak prompt"]
        W["Summarise this note: ..."]
    end
    subgraph Strong["strong prompt"]
        R["role (system instruction)"]
        D["&lt;note&gt; data first"]
        X["&lt;examples&gt; 3 samples"]
        I["&lt;instructions&gt; format + why, last"]
        R --> D --> X --> I
    end
    Weak --> M["Gemini"]
    Strong --> M
    M --> C["check_card(): pass / fail per note"]
```

## Already know this?
This is the first lesson of Part 3. If you can answer all of these, skip to lesson 26 (the agent loop).
1. Name four things you would add to a one-line prompt to make its output reliable.
2. Why is "reply in JSON" in a prompt weaker than a JSON Schema passed to the API?
3. A model's JSON fails your Pydantic model. What do you send back, and how many times?
4. Walk through one tool call: what the model returns, what your code runs, what you send next.
5. The model asks for two tools in one turn. How do you return both results?

## Step by step
### 1. Start the recorder and set up
Same as lesson 07: the recorder first, then Scout's notes, the key from `.env`, one client and the
pinned model id. Without a key the calls replay from `../data/llm_cassettes/19_prompt-engineering.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("19_prompt-engineering")

LLM mode: replay (19_prompt-engineering.yaml)


Setup from lesson 07: the notes, client and model id, unchanged.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
print(len(NOTES), "notes | model:", MODEL)

6 notes | model: gemini-3.5-flash-lite


### 2. Decide what "good" means before you write a prompt
Scout shows notes as one-line **cards** in a list, like `Mars | Two small moons, Phobos and Deimos`.
The code that builds the list needs: one line, a planet name (or `Other`), a ` | ` separator, at most
10 words after it, and no full stop at the end. We write that as a checker first. A prompt you cannot
score is a prompt you cannot improve.

In [3]:
import re

PLANETS = {"Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune", "Other"}

def check_card(text: str) -> list[str]:
    """Return the list of rules a reply breaks (empty list = usable card)."""
    text = (text or "").strip()
    if "\n" in text:
        return ["more than one line"]
    m = re.fullmatch(r"(\w+) \| (.+)", text)
    if not m:
        return ["not 'Planet | summary'"]
    problems = []
    if m.group(1) not in PLANETS:
        problems.append(f"unknown label {m.group(1)!r}")
    if len(m.group(2).split()) > 10:
        problems.append("summary over 10 words")
    if text.endswith("."):
        problems.append("ends with a full stop")
    return problems

for sample in ["Mars | Two small moons, Phobos and Deimos", "Mars has two small moons.", "Pluto | Small"]:
    print(f"{sample!r:45} -> {check_card(sample) or 'ok'}")

'Mars | Two small moons, Phobos and Deimos'   -> ok
'Mars has two small moons.'                   -> ["not 'Planet | summary'"]
'Pluto | Small'                               -> ["unknown label 'Pluto'"]


The checker is plain Python and runs offline. It is a tiny **eval**: lesson 48 builds real ones.

### 3. The weak prompt
This is the prompt most people write first. We run it on notes 1 to 5.
One small helper sends a prompt and returns the reply text and the input token count.

In [4]:
TEST_NOTES = NOTES[:5]

def ask(prompt: str, system: str | None = None) -> tuple[str, int]:
    config = types.GenerateContentConfig(system_instruction=system, max_output_tokens=200,
                                         automatic_function_calling=NO_AFC)
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return (resp.text or "").strip(), resp.usage_metadata.prompt_token_count

def weak_prompt(note: dict) -> str:
    return f"Summarise this note: {note['text']}"

print(weak_prompt(NOTES[0]))

Summarise this note: Mars has two small moons, Phobos and Deimos.


Five calls, one per note. We keep each reply and its input token count.

In [5]:
weak = [ask(weak_prompt(n)) for n in TEST_NOTES]
for n, (text, _) in zip(TEST_NOTES, weak):
    print(f"note {n['id']}: {text!r}")

note 1: 'Mars has two small moons: Phobos and Deimos.'
note 2: 'Venus rotates in the opposite direction (retrograde) to most other planets.'
note 3: '**Summary:** \nA Martian day is approximately 24 hours and 39 minutes long.'
note 4: 'Jupiter is the biggest planet in our solar system.'
note 5: '**Summary:** The Perseverance rover landed on Mars in February 2021.'


Now score them. Each reply either passes every rule or lists what it broke.

In [6]:
def score(results: list[tuple[str, int]]) -> int:
    passed = 0
    for n, (text, _) in zip(TEST_NOTES, results):
        problems = check_card(text)
        passed += not problems
        print(f"note {n['id']}: {'PASS' if not problems else 'FAIL ' + ', '.join(problems)}")
    print(f"=> {passed}/{len(results)} usable cards")
    return passed

weak_score = score(weak)

note 1: FAIL not 'Planet | summary'
note 2: FAIL not 'Planet | summary'
note 3: FAIL more than one line
note 4: FAIL not 'Planet | summary'
note 5: FAIL not 'Planet | summary'
=> 0/5 usable cards


The replies are reasonable summaries, but not one is a card: whole sentences with full stops, and two
start with a Markdown `**Summary:**` label. The model did what we asked; we never said what we wanted.

### 4. Clear instructions, with the reason
Say exactly what to produce, and why. The reason helps the model handle cases the rules do not
cover: knowing the line goes into a list explains why there must be no extra words around it.

In [7]:
INSTRUCTIONS = """Write one card for the note above, for a list in Scout's notes app.
Format: the planet the note is about, then " | ", then a summary of at most 10 words.
- Use one of: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune, Other.
- No full stop at the end, no quotes, no text before or after the card.
The app splits each line on " | ", so anything else breaks the list."""
print(INSTRUCTIONS)

Write one card for the note above, for a list in Scout's notes app.
Format: the planet the note is about, then " | ", then a summary of at most 10 words.
- Use one of: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune, Other.
- No full stop at the end, no quotes, no text before or after the card.
The app splits each line on " | ", so anything else breaks the list.


Notice the format is written as words, not as `{planet} | {summary}`. Common mistake 1 shows why.

### 5. A role
A **role prompt** tells the model who it is and for whom it works. It goes in the system instruction
(lesson 08), so it applies to every request without repeating it in the user's text.

In [8]:
ROLE = ("You are Scout's note indexer. You turn a user's research notes into short, exact index "
        "cards. You never add facts that are not in the note.")
print(ROLE)

You are Scout's note indexer. You turn a user's research notes into short, exact index cards. You never add facts that are not in the note.


### 6. Examples, wrapped in XML tags
**Few-shot examples** show the model finished answers. Use 3 to 5, make them varied (different planets,
one `Other`), and do not reuse your test notes, or you measure memory, not the prompt. **XML tags**
like `<example>` mark where each part starts and ends, so the model never mistakes an example for the
note it must summarise.

In [9]:
EXAMPLES = [
    ("Neptune has the fastest winds measured on any planet.", "Neptune | Fastest winds of any planet"),
    ("Mercury has almost no atmosphere, so its nights are very cold.", "Mercury | Almost no atmosphere, so nights are very cold"),
    ("The James Webb telescope orbits the Sun about 1.5 million km from Earth.", "Other | James Webb telescope orbits 1.5 million km from Earth"),
]
EXAMPLES_XML = "<examples>\n" + "\n".join(
    f"<example>\n<note>{note}</note>\n<card>{card}</card>\n</example>" for note, card in EXAMPLES
) + "\n</examples>"
print(EXAMPLES_XML)
print([check_card(card) for _, card in EXAMPLES])   # our own examples must pass our checker

<examples>
<example>
<note>Neptune has the fastest winds measured on any planet.</note>
<card>Neptune | Fastest winds of any planet</card>
</example>
<example>
<note>Mercury has almost no atmosphere, so its nights are very cold.</note>
<card>Mercury | Almost no atmosphere, so nights are very cold</card>
</example>
<example>
<note>The James Webb telescope orbits the Sun about 1.5 million km from Earth.</note>
<card>Other | James Webb telescope orbits 1.5 million km from Earth</card>
</example>
</examples>
[[], [], []]


### 7. Data first, instructions last
With long inputs, models do better when the data comes first and the request comes at the end
(Anthropic reports up to 30% better answers on long documents). Our notes are short, but the habit
costs nothing and it is the shape your prompts will have when Scout reads whole files.

In [10]:
def strong_prompt(note: dict) -> str:
    return (f"<note>{note['text']}</note>\n\n"
            f"{EXAMPLES_XML}\n\n"
            f"<instructions>\n{INSTRUCTIONS}\n</instructions>")

print(strong_prompt(NOTES[0]))

<note>Mars has two small moons, Phobos and Deimos.</note>

<examples>
<example>
<note>Neptune has the fastest winds measured on any planet.</note>
<card>Neptune | Fastest winds of any planet</card>
</example>
<example>
<note>Mercury has almost no atmosphere, so its nights are very cold.</note>
<card>Mercury | Almost no atmosphere, so nights are very cold</card>
</example>
<example>
<note>The James Webb telescope orbits the Sun about 1.5 million km from Earth.</note>
<card>Other | James Webb telescope orbits 1.5 million km from Earth</card>
</example>
</examples>

<instructions>
Write one card for the note above, for a list in Scout's notes app.
Format: the planet the note is about, then " | ", then a summary of at most 10 words.
- Use one of: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune, Other.
- No full stop at the end, no quotes, no text before or after the card.
The app splits each line on " | ", so anything else breaks the list.
</instructions>


### 8. Run the strong prompt on the same 5 notes
Same model, same notes, same settings. Only the prompt and the role changed.

In [11]:
strong = [ask(strong_prompt(n), system=ROLE) for n in TEST_NOTES]
for n, (text, _) in zip(TEST_NOTES, strong):
    print(f"note {n['id']}: {text!r}")

note 1: 'Mars | Two small moons, Phobos and Deimos'
note 2: 'Venus | Spins backwards compared with most planets'
note 3: 'Mars | A day lasts about 24 hours and 39 minutes'
note 4: 'Jupiter | Largest planet in the solar system'
note 5: 'Mars | Landed on Mars in February 2021'


### 9. Measure the difference
Score, and also compare the input size: a better prompt is longer, and input tokens cost money
(lesson 12).

In [12]:
strong_score = score(strong)
weak_in = sum(t for _, t in weak) / len(weak)
strong_in = sum(t for _, t in strong) / len(strong)
print(f"weak  : {weak_score}/5 usable, ~{weak_in:.0f} input tokens per call")
print(f"strong: {strong_score}/5 usable, ~{strong_in:.0f} input tokens per call")

note 1: PASS
note 2: PASS
note 3: PASS
note 4: PASS
note 5: PASS
=> 5/5 usable cards
weak  : 0/5 usable, ~17 input tokens per call
strong: 5/5 usable, ~312 input tokens per call


0/5 became 5/5 for about 18 times the input tokens (17 to 312). At $0.30 per million input tokens
(Gemini 3.5 Flash-Lite), 1,000 notes cost about 9 cents with the strong prompt, against about half a
cent with the weak one. We changed four things at once, so we don't know yet which one did the work;
the Try-it tests that. Note 5, though: `Mars | Landed on Mars in February 2021` passes
every rule but lost the rover's name. A format checker cannot see meaning; lesson 48 adds checks that can.

### 10. A trick that stopped working: prefill
Older guides "prefill" the reply: they end the message list with an assistant turn such as `Mars |`
so the model must continue from there. On Claude 4.6 and later, Anthropic's API rejects a final
assistant turn with a 400 error. Here is the request shape; we do not send it.

In [13]:
prefill_request = {
    "model": "claude-sonnet-5-5",
    "max_tokens": 50,
    "messages": [
        {"role": "user", "content": strong_prompt(NOTES[0])},
        {"role": "assistant", "content": "Mars |"},   # prefill: rejected (400) on Claude 4.6+
    ],
}
print("last turn role:", prefill_request["messages"][-1]["role"])

last turn role: assistant


Use what you just built instead (clear format, examples, XML tags), and for strict formats, the
structured output of lesson 20.

## At work
- **Prompts are code.** Teams keep them in the repository next to the code that calls them, review
  changes, and pin the model snapshot they were tuned on; a prompt that works on one model can drift
  on the next. → lessons 17 and 53
- **No prompt change ships without an eval.** A fixed set of inputs plus a checker like `check_card`
  runs on every change; "looks better on the one I tried" is how regressions reach users. → lesson 48
- **Prefill removal broke format tricks.** Code that forced JSON by prefilling `{` started returning
  400s on Claude 4.6+; the fix is API-level structured output, which validates the format for you. → lesson 20
- **Stable parts first, changing data last.** Long fixed instructions and examples placed in the same
  order on every call are what prompt caching reuses, so prompt layout is also a cost decision. → lesson 41

## What just happened
- We defined "usable" first, as `check_card`, a checker that runs offline.
- The weak prompt produced fine summaries but no usable cards: it never said what format we needed.
- The strong prompt added clear instructions with a reason, a role in the system instruction, three
  varied examples in `<example>` tags, and the note first with instructions last.
- On the same 5 notes it went from 0/5 to 5/5 usable cards, for about 18 times the input tokens.
- Passing the checker is not the same as a good summary: note 5's card dropped the rover's name.
- Prefill, the old way to force a format, now returns 400 on Claude 4.6+.

## Common mistakes
**1. A template with `{placeholders}` inside the prompt text.** Writing the format as
`{planet} | {summary}` and then filling the note with `.format()` makes Python look for `planet` too.

In [14]:
TEMPLATE = "<note>{note}</note>\nReply as {planet} | {summary}, at most 10 words."
print(TEMPLATE.format(note=NOTES[0]["text"]))

KeyError: 'planet'

Fix: describe the format in words (as in step 4), or double the braces that are meant literally:
`{{planet}}`. Better still, build prompts with small functions like `strong_prompt`.

In [15]:
TEMPLATE = "<note>{note}</note>\nReply as {{planet}} | {{summary}}, at most 10 words."
print(TEMPLATE.format(note=NOTES[0]["text"]))

<note>Mars has two small moons, Phobos and Deimos.</note>
Reply as {planet} | {summary}, at most 10 words.


**2. Judging a prompt by one reply.** One good-looking answer proves nothing; models vary (lesson 04).
Assert on the whole test set and the weak prompt's real failure rate shows at once.

In [16]:
failures = {n["id"]: check_card(t) for n, (t, _) in zip(TEST_NOTES, weak) if check_card(t)}
assert not failures, f"{len(failures)}/5 replies unusable: {failures}"

AssertionError: 5/5 replies unusable: {1: ["not 'Planet | summary'"], 2: ["not 'Planet | summary'"], 3: ['more than one line'], 4: ["not 'Planet | summary'"], 5: ["not 'Planet | summary'"]}

Fix: always score a prompt on several inputs, and keep the inputs fixed so two prompts are compared
on the same notes, as in step 9.

In [17]:
failures = {n["id"]: check_card(t) for n, (t, _) in zip(TEST_NOTES, strong) if check_card(t)}
print(f"strong prompt: {5 - len(failures)}/5 usable, failures: {failures}")

strong prompt: 5/5 usable, failures: {}


## Try it
Which part of the strong prompt matters most? Write `no_examples_prompt(note)`: the strong prompt
without the `<examples>` block. Run it (with the same `ROLE`) on `TEST_NOTES` and score the replies
with `score`. Compare with `strong_score`: did any rule start failing?

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **prompt engineering** — writing and testing the text you send a model so its output is reliably what your code needs.
- **role prompt** — a line, usually in the system instruction, that tells the model who it is and whom it works for.
- **few-shot examples** — a handful (3 to 5) of sample inputs with correct outputs placed in the prompt, so the model copies the pattern.
- **XML tags (in prompts)** — labels like `<note>` and `<example>` that mark where each part of a prompt starts and ends.
- **data-first ordering** — putting long input data at the top of a prompt and the instructions or question at the end.
- **prefill** — starting the model's reply for it with a final assistant turn; rejected with a 400 error on Claude 4.6 and later.
- **eval** — a fixed set of inputs plus a checker that scores a model's outputs, used to compare prompts or models.